In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("covid_toy.csv")

In [3]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [5]:
df["cough"].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [6]:
df["gender"].value_counts()

gender
Female    59
Male      41
Name: count, dtype: int64

In [7]:
df["city"].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [8]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    object 
 2   fever      90 non-null     float64
 3   cough      100 non-null    object 
 4   city       100 non-null    object 
 5   has_covid  100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB


In [9]:
# from above data we conclude that
# age > numerical column > no transformation because column is good.
# gender > nominal categorical column > Using OneHotEncoding to transform this column.
# fever > numerical column > using SimpleImputer to transform this column because there are some null values.
# cough > Ordinal categorical column > Using OrdinalEncoder to transform this column.
# city > nominal categorical column > Using OneHotEncoding to transform this column.

In [10]:
from sklearn.model_selection import train_test_split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(df.drop("has_covid", axis=1), df["has_covid"], test_size=0.2, random_state=0)

In [15]:
X_train.shape

(80, 5)

In [16]:
X_test.shape

(20, 5)

In [23]:
X_train.head(3)

,age,gender,fever,cough,city
43,22,Female,99.0,Mild,Bangalore
62,56,Female,104.0,Strong,Bangalore
3,31,Female,98.0,Mild,Kolkata


# Apply the above transformation one by one on columns.

In [22]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

In [28]:
# for fever column we will apply SimpleImputer
si = SimpleImputer()
X_train_fev = si.fit_transform(X_train[["fever"]])
X_test_fev = si.fit_transform(X_test[["fever"]])

X_train_fev.shape

(80, 1)

In [36]:
# for gender and city column we will apply OneHotEncoder
ohe = OneHotEncoder(drop = "first", sparse_output =False)
X_train_gc = ohe.fit_transform(X_train[["gender", "city"]])
X_test_gc = ohe.fit_transform(X_test[["gender", "city"]])

X_train_gc.shape

(80, 4)

In [60]:
# for cough column we will apply Ordinal Encoding
oe = OrdinalEncoder(categories=[["Mild", "Strong"]])
X_train_cough = oe.fit_transform(X_train[["cough"]])
X_test_cough = oe.fit_transform(X_test[["cough"]])

X_train_cough.shape

(80, 1)

In [49]:
train_age = X_train.drop(columns=["gender", "fever", "cough", "city"]).values
test_age = X_test.drop(columns=["gender", "fever", "cough", "city"]).values

In [51]:
train_age.shape, test_age.shape

((80, 1), (20, 1))

In [53]:
X_train_transformed = np.concatenate((train_age, X_train_fev, X_train_gc, X_train_cough), axis = 1)
X_test_transformed = np.concatenate((test_age, X_test_fev, X_test_gc, X_test_cough), axis = 1)

In [55]:
X_train_transformed.shape

(80, 7)

In [56]:
X_test_transformed.shape

(20, 7)

# Apply above process by using Column transformer

In [57]:
from sklearn.compose import ColumnTransformer

In [62]:
ct = ColumnTransformer(transformers=[
                       ("tnf1",SimpleImputer(), ["fever"]),
                       ("tnf2",OneHotEncoder(drop = "first", sparse_output =False), ["gender", "city"]),
                       ("tnf3",OrdinalEncoder(categories=[["Mild", "Strong"]]), ["cough"])]
                       , remainder = "passthrough")  

In [63]:
ct.fit_transform(X_train)

array([[ 99.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  22.        ],
       [104.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  56.        ],
       [ 98.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  31.        ],
       [104.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   1.        ,  75.        ],
       [ 99.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  72.        ],
       [ 99.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  66.        ],
       [101.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  14.        ],
       [ 98.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   1.        ,  10.        ],
       [ 98.        ,   1.        ,   0.        ,   1.        ,
          0.    

In [64]:
ct.fit_transform(X_train).shape

(80, 7)

In [65]:
ct.fit_transform(X_test)

array([[100.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  19.        ],
       [104.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  25.        ],
       [101.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  42.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  81.        ],
       [102.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,   5.        ],
       [100.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  27.        ],
       [103.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  69.        ],
       [ 98.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   1.        ,  34.        ],
       [ 99.        ,   0.        ,   0.        ,   0.        ,
          1.    

In [66]:
ct.fit_transform(X_test).shape

(20, 7)

In [77]:
cars_df = pd.read_csv("cars.csv") 

In [79]:
cars_df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [80]:
cars_df.shape

(8128, 5)

In [82]:
cars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   brand          8128 non-null   object
 1   km_driven      8128 non-null   int64 
 2   fuel           8128 non-null   object
 3   owner          8128 non-null   object
 4   selling_price  8128 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 317.6+ KB


In [83]:
cars_df["fuel"].value_counts() # onehotencoding

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

In [85]:
cars_df["owner"].value_counts() # ordinalencoder

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

In [87]:
X_train, X_test, y_train, y_test = train_test_split(cars_df.drop(["selling_price"], axis =1), cars_df["selling_price"], test_size=0.2, random_state=0)

In [99]:
transformer = ColumnTransformer(transformers=[
    ("trf1", OneHotEncoder(drop="first", sparse_output=False), ["fuel"]),
    ("trf2", OrdinalEncoder(categories=[["Test Drive Car", "Fourth & Above Owner","Third Owner", "Second Owner", "First Owner" ]]), ["owner"])
], remainder="passthrough")

In [100]:
transformer.fit_transform(X_train)

array([[0.0, 1.0, 0.0, 4.0, 'Hyundai', 60000],
       [1.0, 0.0, 0.0, 2.0, 'Tata', 150000],
       [1.0, 0.0, 0.0, 3.0, 'Hyundai', 110000],
       ...,
       [0.0, 0.0, 1.0, 3.0, 'Hyundai', 90000],
       [1.0, 0.0, 0.0, 4.0, 'Volkswagen', 90000],
       [0.0, 0.0, 1.0, 4.0, 'Hyundai', 110000]], dtype=object)

In [93]:
transformer.fit_transform(X_train).shape

(6502, 6)

In [95]:
transformer.fit_transform(X_test)

array([[1.0, 0.0, 0.0, 4.0, 'Hyundai', 40000],
       [1.0, 0.0, 0.0, 4.0, 'Mahindra', 70000],
       [0.0, 0.0, 1.0, 4.0, 'Maruti', 5000],
       ...,
       [1.0, 0.0, 0.0, 4.0, 'Maruti', 40000],
       [0.0, 0.0, 1.0, 4.0, 'Hyundai', 2350],
       [1.0, 0.0, 0.0, 3.0, 'Hyundai', 80000]], dtype=object)

In [96]:
transformer.fit_transform(X_test).shape

(1626, 6)

In [94]:
X_train

,brand,km_driven,fuel,owner
3042,Hyundai,60000,LPG,First Owner
1520,Tata,150000,Diesel,Third Owner
2611,Hyundai,110000,Diesel,Second Owner
3544,Mahindra,28000,Diesel,Second Owner
4138,Maruti,15000,Petrol,First Owner
...,...,...,...,...
4931,Tata,70000,Diesel,Third Owner
3264,Ford,100000,Diesel,Second Owner
1653,Hyundai,90000,Petrol,Second Owner
2607,Volkswagen,90000,Diesel,First Owner


In [103]:
"trf1"._class

AttributeError: 'str' object has no attribute '_class'